# 🤖 Inference Comparison: Base vs DAPT Fine-Tuned Model
This notebook compares the outputs of the base DeepSeek model and your domain-adapted version using real university-related questions.

# # 🤖 Inference Comparison: SFT Fine-Tuned Models
# This notebook compares the outputs of:
# 1. Base DeepSeek 8B model
# 2. Base 8B model fine-tuned with SFT (QLoRA)
# 3. DAPT 8B model fine-tuned with SFT (QLoRA)
# 4. DAPT 1.3B model fine-tuned with SFT (FP32)
# Models are loaded one by one to conserve VRAM.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from peft import PeftModel 
import pandas as pd
import torch
import gc 
import os 
import time

# === CONFIG ===
BASE_MODEL_8B = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
SFT_BASE_ADAPTER_PATH_8B = "sft_base_8b_qlora_output"
SFT_DAPT_ADAPTER_PATH_8B = "sft_dapt_8b_qlora_output"
SFT_DAPT_MODEL_PATH_1_3B = "sft_dapt_1.3b_fp32_output"

# --- Other Files ---
QUESTIONS_FILE = "../healthandsafety_eval_questions_extended.txt"

# --- Hardware & Inference Settings ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LOAD_IN_4BIT_INFERENCE_8B = True
compute_dtype_4bit = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
dtype_1_3b = torch.float32

In [ ]:
# --- Function to clear memory ---
def clear_memory():
    """Clears GPU cache and runs Python garbage collector."""
    print("🧹 Clearing CUDA cache and collecting garbage...")
    if torch.cuda.is_available():

        torch.cuda.empty_cache()
    gc.collect()
    time.sleep(1)
    print("✅ Memory cleared.")

# --- Configure Quantization ---
bnb_config_inference_8b = None
if LOAD_IN_4BIT_INFERENCE_8B:
    print(f"⚙️ Configuring 4-bit quantization for 8B inference (compute dtype: {compute_dtype_4bit})...")
    bnb_config_inference_8b = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype_4bit,
    )

# --- Check if paths exist ---
model_paths_exist = {
    'sft_base_8b': os.path.exists(SFT_BASE_ADAPTER_PATH_8B),
    'sft_dapt_8b': os.path.exists(SFT_DAPT_ADAPTER_PATH_8B),
    'sft_dapt_1.3b': os.path.exists(SFT_DAPT_MODEL_PATH_1_3B)
}
if not model_paths_exist['sft_base_8b']:
    print(f"⚠️ WARNING: SFT Base 8B adapter path not found: {SFT_BASE_ADAPTER_PATH_8B}")
if not model_paths_exist['sft_dapt_8b']:
    print(f"⚠️ WARNING: SFT DAPT 8B adapter path not found: {SFT_DAPT_ADAPTER_PATH_8B}")
if not model_paths_exist['sft_dapt_1.3b']:
    print(f"⚠️ WARNING: SFT DAPT 1.3B model path not found: {SFT_DAPT_MODEL_PATH_1_3B}")

In [ ]:
# Load evaluation questions
try:
    questions_path = os.path.join("..", "healthandsafety_eval_questions_extended.txt")
    if not os.path.exists(questions_path):
         questions_path = "healthandsafety_eval_questions_extended.txt"

    with open(questions_path, "r", encoding="utf-8") as f:
        questions = [line.strip() for line in f if line.strip()]
    print(f"Loaded {len(questions)} questions from {questions_path}")
    print("First 3 questions:", questions[:3])
except FileNotFoundError:
    print(f"❌ Error: Questions file not found at '{questions_path}' or in current directory.")
    questions = []
except Exception as e:
    print(f"❌ Error reading questions file: {e}")
    questions = []

In [ ]:
# Define models to evaluate
models_to_evaluate = [
    {"key": "base_8b", "load_adapters": False, "adapter_path": None, "is_peft": False, "model_path": BASE_MODEL_8B, "column_name": "Base Model (8B)"},
    {"key": "sft_base_8b", "load_adapters": True, "adapter_path": SFT_BASE_ADAPTER_PATH_8B, "is_peft": True, "model_path": BASE_MODEL_8B, "column_name": "SFT Base (8B QLoRA)"},
    {"key": "sft_dapt_8b", "load_adapters": True, "adapter_path": SFT_DAPT_ADAPTER_PATH_8B, "is_peft": True, "model_path": BASE_MODEL_8B, "column_name": "SFT DAPT (8B QLoRA)"},
    {"key": "sft_dapt_1.3b", "load_adapters": False, "adapter_path": None, "is_peft": False, "model_path": SFT_DAPT_MODEL_PATH_1_3B, "column_name": "SFT DAPT (1.3B FP32)"}
]

# Dictionary to store results for each model
all_results = {}

# --- Inference Loop (Model by Model) ---
print("🚀 Starting SFT inference comparison (one model at a time)...")

if not questions:
    print("⚠️ No questions loaded, skipping inference loop.")
else:
    # --- Define Instruction Prompt ---
    instruction = (
        "You are a member of the University of Bradford staff. "
        "Answer student questions based only on the information you’ve been trained on. "
        "If you are unsure, say you don’t know. Do not make up answers.\n\n"
        "Question: {question}\n"
        "Answer:"
    )

    # --- Loop through each model configuration ---
    for model_config in models_to_evaluate:
        model_key = model_config["key"]
        adapter_path = model_config["adapter_path"]
        load_adapters = model_config["load_adapters"]
        column_name = model_config["column_name"]
        is_peft = model_config["is_peft"]
        model_path = model_config["model_path"] l

        if load_adapters and not model_paths_exist[model_key]:
            print(f"\n--- Skipping model '{column_name}' (adapter path not found) ---")
            all_results[column_name] = ["Model/Adapters not found"] * len(questions)
            continue
        if not load_adapters and model_key != "base_8b" and not model_paths_exist[model_key]:
             print(f"\n--- Skipping model '{column_name}' (model path not found: {model_path}) ---")
             all_results[column_name] = ["Model/Adapters not found"] * len(questions)
             continue

        print(f"\n--- Loading Model: {column_name} ---")
        model = None
        tokenizer = None
        pipe = None
        model_answers = []

        try:
            quant_config = bnb_config_inference_8b if "8b" in model_key else None
            load_dtype = "auto" if "8b" in model_key else dtype_1_3b

            tokenizer_path = BASE_MODEL_8B if is_peft or model_key == "base_8b" else model_path
            print(f"Loading tokenizer from: {tokenizer_path}")
            tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, trust_remote_code=True)
            if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
            default_pad_token_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 50256 # Fallback
            print(f"Loading model from: {model_path} with dtype: {load_dtype}")
            model = AutoModelForCausalLM.from_pretrained(
                model_path,
                quantization_config=quant_config,
                device_map="auto",
                trust_remote_code=True,
                torch_dtype=load_dtype 
            )

            # Apply adapters if needed (for PEFT models)
            if load_adapters:
                print(f"Applying adapters from {adapter_path}...")
                model = PeftModel.from_pretrained(model, adapter_path)
                print("Adapters applied.")

            # Create pipeline
            print(f"Creating pipeline for {column_name}...")
            pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
            print(f"✅ Model {column_name} loaded successfully.")

            # --- Generate answers for all questions with this model ---
            generation_params = {
                "max_new_tokens": 150,
                "do_sample": False,
                "pad_token_id": default_pad_token_id,
                "eos_token_id": default_pad_token_id
            }

            for i, q in enumerate(questions):
                print(f"  Processing question {i+1}/{len(questions)} for {column_name}...")
                # Use the instruction defined outside the model loop
                prompt = instruction.format(question=q)
                try:
                    # Generate text using the pipeline
                    output = pipe(prompt, **generation_params)
                    # Extract generated text
                    if isinstance(output, list) and len(output) > 0 and isinstance(output[0], dict) and "generated_text" in output[0]:
                        output_full = output[0]["generated_text"]
                    else:
                        print(f"  ⚠️ Unexpected output format from pipeline: {output}")
                        model_answers.append("ERROR: Unexpected output format")
                        continue

                    # Extract text after "Answer:"
                    answer_part = output_full.split("Answer:")
                    if len(answer_part) > 1:
                        answer = answer_part[1].strip()
                    else:
                        prompt_base = prompt.split("Answer:")[0]
                        answer = output_full.replace(prompt_base + "Answer:", "").strip() 
                    model_answers.append(answer)

                except Exception as e_gen:
                    print(f"  ❌ Error generating answer for question {i+1} with {column_name}: {e_gen}")
                    model_answers.append(f"ERROR: {e_gen}")

            all_results[column_name] = model_answers

        except Exception as e_load:
            print(f"❌ Error loading or processing model {column_name}: {e_load}")
            import traceback
            traceback.print_exc()
            # Fill results with error message if loading failed
            all_results[column_name] = [f"ERROR loading model"] * len(questions)
        finally:
            print(f"--- Unloading Model: {column_name} ---")
            if 'pipe' in locals() and pipe is not None: del pipe
            if 'model' in locals() and model is not None: del model
            if 'tokenizer' in locals() and tokenizer is not None: del tokenizer
            clear_memory()
            print(f"✅ Model {column_name} unloaded.")

    print("\n✅ All models processed.")

# --- Combine Results into DataFrame ---
if questions:
    df_data = {"Question": questions}
    for model_config in models_to_evaluate:
        col_name = model_config["column_name"]
        if col_name in all_results:
             if len(all_results[col_name]) == len(questions):
                 df_data[col_name] = all_results[col_name]
             else:
                 print(f"⚠️ Warning: Mismatch in number of answers for {col_name}. Expected {len(questions)}, got {len(all_results[col_name])}. Filling with errors.")
                 df_data[col_name] = ["ERROR: Answer list length mismatch"] * len(questions)
        else:
             df_data[col_name] = ["Model failed to load/run"] * len(questions)


    df = pd.DataFrame(df_data)

    column_order = ["Question", "Base Model (8B)", "SFT Base (8B QLoRA)", "SFT DAPT (8B QLoRA)", "SFT DAPT (1.3B FP32)"]
    existing_columns = [col for col in column_order if col in df.columns]
    if existing_columns:
        df = df[existing_columns]
        print("\n--- Comparison Results ---")
        print(df.to_markdown(index=False))
    else:
        print("\n⚠️ No results generated or models loaded successfully.")
else:
    print("\n⚠️ No questions loaded, cannot generate results DataFrame.")
    df = pd.DataFrame()

In [ ]:
if not df.empty:
    try:
        output_csv_file = "sft_model_comparison_output.csv"
        df.to_csv(output_csv_file, index=False)
        print(f"\n✅ Results saved to {output_csv_file}")
    except Exception as e:
        print(f"❌ Error saving results to CSV: {e}")
else:
    print("\n⚠️ DataFrame is empty, skipping CSV save.")
